# Quantum Adders Workbook

What is this workbook? A workbook is a collection of problems, accompanied by solutions to them. The explanations focus on the logical steps required to solve a problem; they illustrate the concepts that need to be applied to come up with a solution to the problem, explaining the mathematical steps required.

This workbook describes the solutions to the problems offered in the Quantum Adders kata. Since the problems involve code implementations of the solutions, the explanations also cover some elements of Workbench that might be non-obvious for a first-time user.

In [ ]:
from psiqdk.workbench import Qubits, QUInt, QInt, Qubrick

### Problem 1.1. In-place sum of two bits

What we're looking is for $a$ to become $a \oplus b$, that is, the sum of bits $a$ and $b$ modulo $2$. This would implement in-place summation of 2 bits.

And we know a gate that allows us to do this: the CNOT gate leaves its control bit unchanged and transforms its target bit to the XOR of the two bits. If we apply a CNOT gate to qubit $b$ as the control and qubit $a$ as the target, it will do exactly what we're looking for!

In [ ]:
def sum_two_bits(a: QUInt | Qubits, b: QUInt | Qubits) -> None:
    a.x(cond=b)

### Problem 1.2. Carry of two bits

For the carry bit of the sum of two bits to be $1$, the sum has to be $2$ or greater, which is possible only if both input bits are $1$. This means that we need to flip the carry bit if $a = 1$ and $b = 1$, or, equivalently, $a \textrm{ AND } b = 1$. This can be achieved using a Toffoli gate - a controlled X gate with both $a$ and $b$ as control qubits.

In [ ]:
def carry_two_bits(a: QUInt | Qubits, b: QUInt | Qubits, c: QUInt | Qubits) -> None:
    c.x(cond=a | b)

### Problem 1.3. In-place sum of three bits

Our goal here is similar to that in problem 1.1: we need to transform qubit $a$ into $a \oplus b \oplus c$ without modifying $b$ and $c$. If we rewrite the sum as $(a \oplus b) \oplus c$, we can calculate it in two steps:

1. Add $b$ to $a$ in-place.
2. Add $c$ to the result in-place.

Each step is a CNOT gate, with $a$ as the target and $b$ and $c$ as controls, respectively.

In [ ]:
def sum_three_bits(a: QUInt | Qubits, b: QUInt | Qubits, c: QUInt | Qubits) -> None:
    a.x(cond=b)
    a.x(cond=c)

### Problem 1.4. Carry of three bits

The resulting value of carry bit $d$ should be the majority of bits $a$, $b$, and $c$, that is, $1$ if two or three of them are $1$, and $0$ otherwise.

The majority of three inputs $a$, $b$, and $c$ can be calculated using the following formula:

$$(a \cdot b) \oplus (a \cdot c) \oplus (b \cdot c)$$

Indeed, you can check that if one or none of the inputs are $1$, each of the expressions in the brackets is $0$, and their sum is $0$ as well. If exactly two inputs are $1$, exactly one of the expressions in the brackets is $1$, and their sum is $1$. Finally, if all three inputs are $1$, all three expressions in the brackets are $1$, and their sum modulo $2$ is $1$ as well.

Similarly to problem 1.2, we can use the CCNOT gate with pairs of inputs as controls and the qubit $d$ as the target to add pairwise 
ANDs of the inputs to the target qubit.

In [ ]:
def carry_three_bits(a: QUInt | Qubits, b: QUInt | Qubits, c: QUInt | Qubits, d: QUInt | Qubits) -> None:
    d.x(a | b)
    d.x(a | c)
    d.x(b | c)

### Problem 1.5. Two-bit ripple-carry adder

First, we will allocate a qubit to store the carry bit from adding the least significant bits.

As we have to perform in-place addition, we will first have to figure out the value of this carry bit. We can do that using the function `carry_two_bits` from problem 1.2 for the least significant bits.

After that, we can compute the sum of the most significant bits and the carry bit; we can use the function `sum_three_bits` from problem 1.3 for this.

The next step is uncomputing the carry bit using the function `carry_two_bits` again. (If we compute the sum of the least significant bits first, the value stored in `a[0]` will change, which will lead to incorrect uncomputation.)

Finally, we can compute the sum of the least significant bits using the function `sum_two_bits` from problem 1.1 and release the carry bit.

In [ ]:
class RippleCarryAdderTwoBit(Qubrick):
    def _compute(self, a: QUInt, b: QUInt, **kwargs) -> None:
        """Add register b to register a in-place."""
        carry = self.alloc_temp_qreg(1, "carry")

        # Compute carry of the least significant bits
        carry_two_bits(a[0], b[0], carry)

        # If we wanted to implement non-modular addition,
        # we'd compute the carry of the most significant bit here
        # using carry_three_bits and another carry bit

        # Compute sum of the most significant bits (include carry)
        sum_three_bits(a[1], b[1], carry)
        
        # Uncompute carry
        carry_two_bits(a[0], b[0], carry)
        
        # Compute sum of the least significant bits
        sum_two_bits(a[0], b[0])

        # Release the carry qubit
        carry.release()

### Problem 1.6. Ripple-carry adder

First, we will allocate a qubit register of length $N - 1$ to store the internal carry bits.

As we have to perform in-place addition, we will first have to figure out all the internal carries. (If we compute any of the sums instead of computing carries in the first step, it will change the state of $a$ due to addition happening in-place, and we will get the wrong carry.) We can get the least significant carry bit using the function `carry_two_bits` from problem 1.2 and the rest of the carry bits (from least to most significant) - using the function `carry_three_bits` from problem 1.4.

After this, we need to perform in-place addition and store the results in the register $a$ while also uncomputing the carry register to release it at the end of the computation. For this, we will loop from the last to the first qubit, i.e., most to least significant bit, uncompute the carries using the same function `carry_three_bits` (which is conveniently self-adjoint) and perform in-place addition using the function `sum_three_bits` from problem 1.3.

Finally, for the least significant bit, we will uncompute the carry bit first (using the function `carry_two_bits`) and then compute the sum using the function `sum_two_bits` from problem 1.1.

> While computing the carry bits, we go from index $0$ to $N - 2$, as we require the previous carry to get the new carry. But for the in-place addition and uncomputation of carry bits, we go in the opposite direction, from the index $N - 1$ to $0$, because we require the previous carry to compute the sum, and on addition the qubits of the register $a$ change.

In [ ]:
class RippleCarryAdder(Qubrick):
    def _compute(self, a: QUInt, b: QUInt, **kwargs) -> None:
        """Add register b to register a in-place."""
        n = len(a)
        carry = self.alloc_temp_qreg(n - 1, "carry")
        
        # Compute carry of the least significant bits
        carry_two_bits(a[0], b[0], carry[0])

        # Compute carry of the next N - 2 bits
        for i in range(1, n - 1):
            carry_three_bits(a[i], b[i], carry[i - 1], carry[i])

        # If we wanted to implement non-modular addition,
        # we'd compute the carry of the most significant bit too
        # using another carry bit that would be part of the output

        # Compute sums (including carries) and uncompute carries
        for i in range(n - 1, 0, -1):
            if i < n - 1:
                carry_three_bits(a[i], b[i], carry[i - 1], carry[i])
            sum_three_bits(a[i], b[i], carry[i - 1])

        # Uncompute carry of the least significant bits
        carry_two_bits(a[0], b[0], carry[0])

        # Compute sum of the least significant bits
        sum_two_bits(a[0], b[0])

        # Release the carry qubits
        carry.release()

### Problem 2.1. Majority gate

As we've seen in problem 1.4, the carry of the sum of three bits is their majority: $1$ if two or three of them are $1$, and $0$ otherwise. The majority gate effectively calculates the carry of three bits while also modifying the other two bits in a way that enables the other building block of the algorithm - uncomputation with addition.

We can transform $\ket{a}\ket{b} \rightarrow \ket{a \oplus b}\ket{b}$ easily using a CNOT gate as long as we do it before we modify the state of the register $b$. Similarly, we can transform $\ket{b}\ket{c} \rightarrow \ket{b}\ket{b \oplus c}$ using another CNOT gate. With the values $a \oplus b$ and $b \oplus c$ in place instead of $a$ and $c$, how can we transform the bit $b$ into the majority?

Let's use the formula from problem 1.4 and rewrite it as follows, keeping in mind that for $b \in \{ 0, 1 \}$ and calculations done module $2$ $-b^2 = b^2 = b$:

$$\textrm{MAJ}(a, b, c) = (a \cdot b) \oplus (a \cdot c) \oplus (b \cdot c) = \left(a \cdot (b \oplus c) \right) \oplus (b \cdot c)$$

$$=\left(a \cdot (b \oplus c) \right) \oplus \left(b \cdot (b + c)\right) \oplus (b \cdot b) = (a \oplus b) \cdot (b \oplus c) \oplus b$$

This expression gives us the last gate we need to use: a CCNOT gate with values $a \oplus b$ and $b \oplus c$ as the controls and $b$ as the target.

In [ ]:
def maj(a: QUInt, b: QUInt, c: QUInt) -> None:
    # a -> a + b
    a.x(cond=b)
    # c -> c + b
    c.x(cond=b)
    # b -> maj
    b.x(cond=a | c)

### Problem 2.2. UnMajority and Add gate

We need to partially uncompute the transformation done by the majority gate, so we can start by retracing the steps of the solution to the previous problem in reverse order.

The last step of the majority gate calculated the majority and stored it in the second qubit; we need to reverse this step, so we can apply the same CCNOT gate to do it:

$$\ket{a \oplus b} \ket{\textrm{carry}(a, b, c)} \ket{b \oplus c} \rightarrow \ket{a \oplus b} \ket{b} \ket{b \oplus c}$$

The second-to-last step of the majority gate calculated $b \oplus c$ and stored it in the third qubit; we need to reverse this step as well using a CNOT gate:

$$\ket{a \oplus b} \ket{b} \ket{b \oplus c} \rightarrow \ket{a \oplus b} \ket{b} \ket{c}$$

Now we need to change the state of the first qubit from $a \oplus b$ to $a \oplus b \oplus c$ instead of $a$ it was originally. To do this, we only need to remember that we've already recovered the bit $c$ in the last qubit; we can add it to the state of the first qubit using a CNOT gate.

In [ ]:
def uma(a: QUInt, b: QUInt, c: QUInt) -> None:
    # carry -> b
    b.x(cond=a | c)
    # b + c -> c
    c.x(cond=b)
    # a + b -> a + b + c
    a.x(cond=c)

### Problem 2.3. One-bit Cuccaro adder

Since we're supposed to use MAJ and UMA primitives, which both act on three qubit registers, we need to allocate a temporary qubit. (This makes one-bit Cuccaro adder less qubit-efficient than one-bit ripple-carry adder, but it will make up for this for larger inputs!)

With that qubit in the $\ket{0}$ state as the third qubit, we can just use the MAJ function, followed by the UMA function, to implement the following transformation:

$$\ket{a} \ket{b} \ket{0} \rightarrow \ket{a \oplus b} \ket{\textrm{carry}(a, b, 0)} \ket{b} \rightarrow \ket{a \oplus b} \ket{b} \ket{0}$$

In [ ]:
class CuccaroAdderOneBit(Qubrick):
    def _maj(self, a: QUInt | Qubits, b: QUInt | Qubits, c: QUInt | Qubits) -> None:
        # a, b, c -> a + b, carry, b + c
        a.x(cond=b)
        c.x(cond=b)
        b.x(cond=a | c)
    def _uma(self, a: QUInt | Qubits, b: QUInt | Qubits, c: QUInt | Qubits) -> None:
        # a + b, carry, b + c -> a + b + c, b, c
        b.x(cond=a | c)
        c.x(cond=b)
        a.x(cond=c)
    def _compute(self, a: QUInt, b: QUInt, **kwargs) -> None:
        """Add register b to register a in-place."""
        c = self.alloc_temp_qreg(1, "c")
        self._maj(a, b, c)
        self._uma(a, b, c)
        c.release()

### Problem 2.4. Two-bit Cuccaro adder

Same as in the previous case, we start by allocating an auxiliary qubit $c$.

Similarly to the ripple-carry adder, we want to start addition by calculating the carry bit for the least significant bit; we can do that using the MAJ function. Now, the qubit `b[0]` holds the carry bit, which we can use to calculate the sum of the most significant bits using the MAJ-UMA sequence on qubits `a[1]`, `b[1]`, and `b[0]`. After this, the qubits `a[1]` and `b[1]` will hold their final values (the most significant bits of the sum and the input $b$, respectively), and the qubit `b[0]` will still hold the carry bit, which we can use to finish the computation of the sum of the least significant bits.

> Note that we use the actual auxiliary qubit only once, for addition of the least significant qubits. The exact implementation of MAJ and UMA functions for the Cuccaro adder can vary, and so can the use of the auxiliary qubit vs the bits of $a$ or $b$ to store the intermediary carry bits. If your solution arrives to the same result using different qubits in different roles, it's fine!

In [ ]:
class CuccaroAdderTwoBit(Qubrick):
    def _maj(self, a: QUInt | Qubits, b: QUInt | Qubits, c: QUInt | Qubits) -> None:
        # a, b, c -> a + b, carry, b + c
        a.x(cond=b)
        c.x(cond=b)
        b.x(cond=a | c)
    def _uma(self, a: QUInt | Qubits, b: QUInt | Qubits, c: QUInt | Qubits) -> None:
        # a + b, carry, b + c -> a + b + c, b, c
        b.x(cond=a | c)
        c.x(cond=b)
        a.x(cond=c)
    def _compute(self, a: QUInt, b: QUInt, **kwargs) -> None:
        """Add register b to register a in-place."""
        c = self.alloc_temp_qreg(1, "c")
        self._maj(a[0], b[0], c)
        self._maj(a[1], b[1], b[0])
        self._uma(a[1], b[1], b[0])
        self._uma(a[0], b[0], c)
        c.release()

### Problem 2.5. Cuccaro adder

We can generalize the solution to the previous problem to an arbitrary number of bits $N$.

1. First, we go from the least significant bits to the most significant bits, using the MAJ function to calculate the carry bits and partial sums of bits.

2. Then, we go in the reverse order, from most to least significant bits, using the UMA function to calculate the sum in $a$ and uncompute the transformations to the bits of $b$.

In [ ]:
class CuccaroAdder(Qubrick):
    def _maj(self, a: QUInt | Qubits, b: QUInt | Qubits, c: QUInt | Qubits) -> None:
        # a, b, c -> a + b, carry, b + c
        a.x(cond=b)
        c.x(cond=b)
        b.x(cond=a | c)
    def _uma(self, a: QUInt | Qubits, b: QUInt | Qubits, c: QUInt | Qubits) -> None:
        # a + b, carry, b + c -> a + b + c, b, c
        b.x(cond=a | c)
        c.x(cond=b)
        a.x(cond=c)
    def _compute(self, a: QUInt, b: QUInt, **kwargs) -> None:
        """Add register b to register a in-place."""
        c = self.alloc_temp_qreg(1, "c")
        n = len(a)
        for i in range(n):
            self._maj(a[i], b[i], c if i == 0 else b[i - 1])
        for i in range(n - 1, -1, -1):
            self._uma(a[i], b[i], c if i == 0 else b[i - 1])
        c.release()

> Copyright (c) 2026 PsiQuantum